# Step 3 Part D: Leland's adjusted-volatility hedging

Leland's idea: instead of hedging with the TRUE volatility, use an ARTIFICIALLY INFLATED volatility in the Black-Scholes delta formula. This makes the hedge trade less aggressively in response to small price wiggles, which reduces transaction costs -- a simple, elegant, closed-form way to account for trading frictions.

Formula for the adjusted volatility:

$$\sigma_{adj}^2 = \sigma^2 \left(1 + \sqrt{\frac{2}{\pi}} \cdot \frac{k}{\sigma \sqrt{\Delta t}}\right)$$

where $\sigma$ is the true (realized/implied) volatility, $k$ is the round-trip proportional transaction cost rate, and $\Delta t$ is the rebalancing interval in years (1 hour here). Note the $+$ sign is for a SHORT option position hedged by BUYING the underlying delta (our setup) -- Leland's adjustment direction depends on whether you're hedging a long or short option position.

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt

train = pd.read_csv("btc_options_train.csv")
train["hour_bucket"] = pd.to_datetime(train["hour_bucket"])
train["sample_date"] = train["hour_bucket"].dt.date

def bs_delta(S, K, T_years, sigma, option_type, r=0.0):
    if T_years <= 0 or sigma <= 0:
        if option_type == "call":
            return 1.0 if S > K else 0.0
        else:
            return -1.0 if S < K else 0.0
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T_years) / (sigma * np.sqrt(T_years))
    return norm.cdf(d1) if option_type == "call" else norm.cdf(d1) - 1.0

def leland_adjusted_vol(sigma, k, dt_years):
    """sigma: true annualized vol (decimal). k: round-trip proportional cost rate. dt_years: rebalancing interval in years."""
    adj_factor = 1 + np.sqrt(2 / np.pi) * (k / (sigma * np.sqrt(dt_years)))
    return sigma * np.sqrt(adj_factor)

counts = train.groupby(["symbol", "sample_date"]).size().sort_values(ascending=False)
proto_symbol, proto_date = counts.index[0]
episode = train[(train["symbol"] == proto_symbol) & (train["sample_date"] == proto_date)].sort_values("hour_bucket").reset_index(drop=True)
episode["T_years"] = episode["time_to_maturity_days"] / 365
episode["iv_decimal"] = episode["mark_iv"] / 100
episode["option_mid_usd"] = episode["mid_price"] * episode["underlying_price"]
episode["option_pnl"] = -episode["option_mid_usd"].diff().fillna(0)

DT_YEARS = 1 / (365 * 24)  # hourly rebalancing

episode["bs_delta"] = episode.apply(
    lambda row: bs_delta(row["underlying_price"], row["strike_price"], row["T_years"], row["iv_decimal"], row["type"]), axis=1
)

# Leland: use each row's own relative_spread as the round-trip cost rate k
episode["leland_sigma"] = episode.apply(
    lambda row: leland_adjusted_vol(row["iv_decimal"], row["relative_spread"], DT_YEARS), axis=1
)
episode["leland_delta"] = episode.apply(
    lambda row: bs_delta(row["underlying_price"], row["strike_price"], row["T_years"], row["leland_sigma"], row["type"]), axis=1
)

episode[["hour_bucket", "iv_decimal", "relative_spread", "leland_sigma", "bs_delta", "leland_delta"]]

## Bug found: transaction cost rate was wrong
We were using the OPTION's relative bid-ask spread (14-22% here) as the transaction cost rate `k` for hedging in BTC. That's the cost of trading the illiquid OPTION, not the cost of trading BTC itself, which is what we actually rebalance with. Real BTC spot/perpetual trading costs are a few basis points, not double-digit percentages. This blew up Leland's formula into absurd volatility values (300%+), making it MORE aggressive instead of less.

Fix: use a fixed, realistic BTC transaction cost assumption (5 basis points round-trip, a reasonable estimate for Deribit's BTC-PERPETUAL market) for BOTH the P&L engine's cost calculation AND as `k` in Leland's formula. The option's own spread stays as a separate, useful feature (e.g. for later liquidity analysis) but is no longer used as the hedging cost rate.

In [ ]:
BTC_TRANSACTION_COST_RATE = 0.0005  # 5 basis points round-trip, a realistic BTC spot/perpetual cost assumption
episode["btc_cost_rate"] = BTC_TRANSACTION_COST_RATE

# Recompute Leland's sigma using the REALISTIC cost rate, not the option's spread
episode["leland_sigma"] = episode.apply(
    lambda row: leland_adjusted_vol(row["iv_decimal"], row["btc_cost_rate"], DT_YEARS), axis=1
)
episode["leland_delta"] = episode.apply(
    lambda row: bs_delta(row["underlying_price"], row["strike_price"], row["T_years"], row["leland_sigma"], row["type"]), axis=1
)
episode[["hour_bucket", "iv_decimal", "leland_sigma", "bs_delta", "leland_delta"]]

## Compare: does Leland's delta move less aggressively than plain BS delta?

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(episode["hour_bucket"], episode["bs_delta"], marker="o", label="Plain BS delta")
ax.plot(episode["hour_bucket"], episode["leland_delta"], marker="o", label="Leland delta")
ax.legend()
ax.set_title("BS delta vs Leland-adjusted delta")
plt.tight_layout()
plt.show()

## Run the same simulation engine, now with Leland delta as target

In [ ]:
def simulate_full_hedge(episode_df, target_delta_col, cost_rate_col="btc_cost_rate"):
    n = len(episode_df)
    position = np.zeros(n)
    trade_size = np.zeros(n)
    transaction_cost = np.zeros(n)
    hedge_pnl = np.zeros(n)

    current_position = 0.0
    for i in range(n):
        target = episode_df[target_delta_col].iloc[i]
        trade = target - current_position
        trade_size[i] = trade

        spot = episode_df["underlying_price"].iloc[i]
        cost_rate = episode_df[cost_rate_col].iloc[i]
        transaction_cost[i] = abs(trade) * spot * (cost_rate / 2)

        if i > 0:
            prev_spot = episode_df["underlying_price"].iloc[i - 1]
            hedge_pnl[i] = current_position * (spot - prev_spot)

        current_position = target
        position[i] = current_position

    result = episode_df.copy()
    result["position"] = position
    result["trade_size"] = trade_size
    result["transaction_cost"] = transaction_cost
    result["hedge_pnl"] = hedge_pnl
    result["total_pnl"] = result["option_pnl"] + result["hedge_pnl"] - result["transaction_cost"]
    result["cumulative_total_pnl"] = result["total_pnl"].cumsum()
    return result

sim_bs = simulate_full_hedge(episode, target_delta_col="bs_delta")
sim_leland = simulate_full_hedge(episode, target_delta_col="leland_delta")

print("=== Plain BS delta ===")
print(f"Total cost: ${sim_bs['transaction_cost'].sum():.4f}")
print(f"Turnover: {sim_bs['trade_size'].abs().sum():.6f} BTC")
print(f"Final total P&L: ${sim_bs['total_pnl'].sum():.4f}")

print("\n=== Leland delta ===")
print(f"Total cost: ${sim_leland['transaction_cost'].sum():.4f}")
print(f"Turnover: {sim_leland['trade_size'].abs().sum():.6f} BTC")
print(f"Final total P&L: ${sim_leland['total_pnl'].sum():.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sim_bs["hour_bucket"], sim_bs["cumulative_total_pnl"], marker="o", label="Plain BS delta")
ax.plot(sim_leland["hour_bucket"], sim_leland["cumulative_total_pnl"], marker="o", label="Leland delta")
ax.axhline(0, color="gray", linestyle="--")
ax.legend()
ax.set_title("Cumulative total P&L: BS vs Leland")
plt.tight_layout()
plt.show()

## Confirm on a near-the-money episode
Deep OTM options have low gamma, so Leland has little turnover to save there. Let's find an ATM episode (moneyness close to 1.0) with good data and confirm Leland's benefit shows up more clearly, as expected.

In [ ]:
train["moneyness_dist"] = (train["moneyness"] - 1.0).abs()
atm_counts = train[train["moneyness_dist"] < 0.05].groupby(["symbol", "sample_date"]).size().sort_values(ascending=False)
atm_symbol, atm_date = atm_counts.index[0]
print(f"ATM episode: {atm_symbol} on {atm_date}, {atm_counts.iloc[0]} rows")

atm_episode = train[(train["symbol"] == atm_symbol) & (train["sample_date"] == atm_date)].sort_values("hour_bucket").reset_index(drop=True)
atm_episode["T_years"] = atm_episode["time_to_maturity_days"] / 365
atm_episode["iv_decimal"] = atm_episode["mark_iv"] / 100
atm_episode["option_mid_usd"] = atm_episode["mid_price"] * atm_episode["underlying_price"]
atm_episode["option_pnl"] = -atm_episode["option_mid_usd"].diff().fillna(0)
atm_episode["btc_cost_rate"] = BTC_TRANSACTION_COST_RATE

atm_episode["bs_delta"] = atm_episode.apply(
    lambda row: bs_delta(row["underlying_price"], row["strike_price"], row["T_years"], row["iv_decimal"], row["type"]), axis=1
)
atm_episode["leland_sigma"] = atm_episode.apply(
    lambda row: leland_adjusted_vol(row["iv_decimal"], row["btc_cost_rate"], DT_YEARS), axis=1
)
atm_episode["leland_delta"] = atm_episode.apply(
    lambda row: bs_delta(row["underlying_price"], row["strike_price"], row["T_years"], row["leland_sigma"], row["type"]), axis=1
)

atm_sim_bs = simulate_full_hedge(atm_episode, target_delta_col="bs_delta", cost_rate_col="btc_cost_rate")
atm_sim_leland = simulate_full_hedge(atm_episode, target_delta_col="leland_delta", cost_rate_col="btc_cost_rate")

print("\n=== ATM: Plain BS ===")
print(f"Cost: ${atm_sim_bs['transaction_cost'].sum():.4f}, Turnover: {atm_sim_bs['trade_size'].abs().sum():.6f}")
print("\n=== ATM: Leland ===")
print(f"Cost: ${atm_sim_leland['transaction_cost'].sum():.4f}, Turnover: {atm_sim_leland['trade_size'].abs().sum():.6f}")